In [12]:
import os
import json
from dotenv import load_dotenv
from IPython.display import display, Markdown, update_display
from scrapper import fetch_website_links_farhan, fetch_website_contents_farhan
from openai import OpenAI

In [13]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith("sk-proj-") and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API Key? Please visit the troubleshooting notebook")

MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [14]:
links = fetch_website_links_farhan("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

In [15]:
link_system_prompt = """
you are provided with a list of links on a webpage.
You are able to check which of these links would be most relevant to include in a brochure about the company,
such as links to an About Page, or a Company page, or Careers/Jobs pages.
You should respond in a JSON as in this example:
{
    "links":[
    {"type": "about page", "url":"https://full.url/goes/here/about"},
    {"type": "career page","url":"https://another.full.url/careers"}
    ]
}
"""

In [16]:
def get_links_user_prompt(url):
    user_prompt = f"""
    Here is the list of links on the website {url} - 
    Please decide which of these links are relevant web links for a brochure about the company,
    respond with the full https URL in json format.
    Do not include terms of Service, Privacy, email links.

    Links (some might be a relative links)
    
    """
    links = fetch_website_links_farhan(url)
    user_prompt+="\n".join(links)
    return user_prompt


In [17]:
print(get_links_user_prompt("https://edwarddonner.com"))


    Here is the list of links on the website https://edwarddonner.com - 
    Please decide which of these links are relevant web links for a brochure about the company,
    respond with the full https URL in json format.
    Do not include terms of Service, Privacy, email links.

    Links (some might be a relative links)

    #wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
http

In [23]:
def select_relevant_links(url):
    print(f"selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)},
        ],
        response_format = {"type":"json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links["links"])} relevant links")
    return links




In [ ]:
select_relevant_links("https://huggingface.co")

selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'career page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'blog page (Chinese)', 'url': 'https://huggingface.co/blog/zh'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'Zhihu page', 'url': 'https://www.zhihu.com/org/huggingface/'}]}

In [25]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents_farhan(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page: \n\n{contents}\n## Relavant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents_farhan(link['url'])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

In [ ]:
brochure_system_prompt = """
You are a assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in Markdown without code blocks.
Include details of a company culture, customers and careers/jobs if you have the information.
"""

In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
    You are looking at a company called {company_name}
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code block. \n\n
    """
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [30]:
get_brochure_user_prompt(company_name="HuggingFace", url="https://huggingface.co")

selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 6 relevant links


'\n    You are looking at a company called HuggingFace\n    Here are the contents of its landing page and other relevant pages;\n    use this information to build a short brochure of the company in markdown without code block. \n\n\n    ## Landing Page: \n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nempero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF\nUpdated\n6 days ago\n•\n1.46M\n•\n1.45k\nzai-org/GLM-5.2\nUpdate

In [33]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)},
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [34]:
create_brochure(company_name="HuggingFace", url="https://huggingface.co")

selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face Company Brochure

---

## About Hugging Face

**Hugging Face** is the dynamic AI community building the future of machine learning. Positioned as the home for machine learning enthusiasts and professionals, Hugging Face offers a vibrant platform where developers, researchers, and organizations collaborate on models, datasets, and AI applications. Their ecosystem empowers users worldwide to create, discover, and innovate in AI and machine learning like never before.

---

## What We Offer

- **Models:** Explore and deploy from over 2 million state-of-the-art machine learning models in various domains, updated regularly by a global community.
- **Datasets:** Access and share more than 500,000 datasets curated for AI training and evaluation.
- **Spaces:** Host and interact with 1 million+ AI-powered applications, ranging from voice chat to image editing and text extraction.
- **Buckets & Storage:** Enterprise-grade storage solutions to host models and datasets efficiently.
- **Enterprise Solutions:** Tailored support with Hugging Face PRO, inference providers, and endpoints designed for business needs.
- **Learning & Community:** Rich documentation, forums, Discord channels, daily AI research paper posts, and open collaboration spaces.

---

## Our Community & Culture

Hugging Face fosters an **open, collaborative, and innovative** community culture. As a collaboration platform, it promotes transparency and sharing across the machine learning ecosystem. Whether you’re a hobbyist, a researcher, or an enterprise, Hugging Face provides the tools and network to build and scale AI technologies responsibly and effectively.

- **Inclusive & Accessible:** Hosting unlimited public models and datasets encourages global participation.
- **Cutting-edge Innovation:** Active engagement with trending models and AI applications.
- **Collaborative Spirit:** Easy-to-use platform for sharing code, datasets, and interactive demos.

---

## Customers & Users

Hugging Face serves a broad user base including:

- **AI Researchers & Developers:** Leverage an extensive hub of models and datasets to accelerate research and development.
- **Enterprises & Teams:** Integrate advanced AI solutions with enterprise support, custom infrastructure, and managed services.
- **Educational Institutions:** Use the platform for teaching, experimentation, and refinement of AI concepts.
- **Open Source Contributors:** Contribute models, datasets, applications, and AI tools to a thriving open-source ecosystem.

---

## Careers at Hugging Face

Join Hugging Face to be part of an AI revolution! The company is continuously growing its **team of engineers, researchers, and community experts** passionate about democratizing AI.

- Work alongside leading AI experts and innovators.
- Engage in a mission-driven culture with open collaboration and cross-functional teamwork.
- Positions often available in machine learning engineering, software development, data science, research, and community management.

Find current openings and learn about the culture on their website.

---

## Brand & Visual Identity

- Signature colors: Vibrant yellow (#FFD21E), warm orange (#FF9D00), and neutral gray (#6B7280).
- Accessible brand assets are available for community and partner use, including logos in multiple formats (.svg, .png, .ai).

---

## Connect With Hugging Face

- **Website:** [huggingface.co](https://huggingface.co)
- **Community:** Active on Discord, GitHub, Forums, and Blog platforms.
- **Explore:** Access models, datasets, and AI applications directly on their platform.

---

*Hugging Face is transforming how the world builds and shares AI. Whether you are exploring machine learning for the first time or building enterprise-grade AI applications, Hugging Face is your partner in innovation.*

In [37]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)},
        ],stream = True
    )
    
    response = ""
    display_handle = display(Markdown(""),display_id=True)
    for chunk in stream:
        response+= chunk.choices[0].delta.content or ''
        update_display(Markdown(response),display_id=display_handle.display_id)

In [38]:
stream_brochure(company_name="HuggingFace", url="https://huggingface.co")

selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the vibrant AI community and platform dedicated to building the future of machine learning. It serves as the central hub where developers, researchers, and enterprises collaborate to create, discover, and share state-of-the-art models, datasets, and machine learning applications.

With over **2 million models**, **500,000+ datasets**, and **1 million+ applications**, Hugging Face empowers innovation across Natural Language Processing (NLP), computer vision, speech, reinforcement learning, and beyond. It enables seamless collaboration and open-source contributions, making machine learning accessible and usable for everyone.

---

## Platform & Solutions

- **Models** – Access and share extensive pre-trained AI models, including popular ones such as Claude Mythos, GLM 5.2, and specialized OCR models.
- **Datasets** – Browse and contribute to hundreds of thousands of structured datasets that fuel ML research and applications.
- **Spaces** – Explore and deploy interactive AI apps running on the platform, including voice chat, image editing, and video generation tools.
- **Enterprise Solutions** – Tailored team and enterprise plans include Hugging Face PRO, enterprise support, inference endpoints, storage buckets, and inference providers to help businesses integrate and scale AI responsibly.
- **Community Tools** – Forums, Discord channels, blogs, and daily research papers keep users connected, informed, and engaged.

---

## Company Culture

Hugging Face thrives on openness, collaboration, and innovation. The company fosters a strong community spirit where knowledge-sharing and collective growth are central. Its culture is rooted in:

- **Open Source Ethos** — Encouraging contributions, transparency, and building technology for the greater good.
- **Community First** — Supporting a dynamic, inclusive ML community of researchers, engineers, hobbyists, and enterprises.
- **Cutting-edge Research & Ethics** — Emphasizing responsible AI development, ethical considerations, and cutting-edge scientific advancements.
- **Continuous Learning** — Providing resources such as blogs, forums, and learning materials to help everyone stay on the frontier of AI.

---

## Customers & Use Cases

Hugging Face supports a diverse customer base ranging from individual developers and startups to large enterprises and research institutions. Its services and models are used in:

- Automating document and image OCR processing
- Developing conversational AI and chatbots
- Enhancing gaming with AI-driven content and behavior modeling
- Video and image generation from text prompts
- Advancing research in NLP, audio processing, computer vision, and reinforcement learning

---

## Careers at Hugging Face

Hugging Face is growing rapidly and looks for talented, passionate individuals to join the team. They offer exciting roles in:

- Machine Learning Engineering
- Research & Development
- Software Engineering
- Enterprise Solutions & Support
- Community Engagement & Developer Relations

The company values innovation, open collaboration, and a shared passion for AI. It provides opportunities to work on impactful projects that reach millions worldwide.

---

## Connect & Learn More

- **Website:** [huggingface.co](https://huggingface.co)
- **Community:** Active forums, Discord, and GitHub repositories
- **Blog:** Insights, case studies, research, and tutorials
- **Enterprise:** Customized AI integration and support for businesses

---

Hugging Face invites you to join the AI revolution — collaborate, build, and shape the future of machine learning with the global community powering it.

---

*Hugging Face - The AI community building the future.*